# Gkatis 2025 $\chi^2$ with TOF-folded $c_0$

Reduced $\chi^2/N$ of **JEFF-4.0**, **JENDL-5**, and **This work (run 77)** against the
out-of-sample Gkatis 2025 $^{56}$Fe elastic angular dataset (EXFOR `27673002`).

The L=0 normalization $c_0$ of **every** library is set the *same* way: from its own
elastic cross section (MF3), Gaussian-folded over the Gkatis time-of-flight energy
resolution,

$$c_0^{\mathrm{lib}}(E)=\langle\sigma_{\mathrm{MF3}}^{\mathrm{lib}}(E')\rangle_{\mathrm{fold}}/(4\pi),$$

instead of the mixed recipe used elsewhere (Kinney/Smith GLS fit for JEFF/JENDL, stored
`nominal_fits.parquet` for This work). This is the existing `folded_c0` methodology
(`scripts/precompute_chi2_folded_c0.py`), here run inline for **run 77** and scoped to
Gkatis. The TOF width uses `compute_sigma_E` (Gkatis $\delta t = 10$ ns treated as $\sigma$).

The table reports the two covariance budgets, matching `tab:gkatis_chi2`:

* **V2** $\;\sigma_\mathrm{exp}$ &mdash; EXFOR-only ($D + uu^\top + vv^\top$, no MF34).
* **V4** $\;\sigma_\mathrm{exp}\oplus\sigma_\mathrm{eval}$ &mdash; EXFOR + dense MF34.

## Configuration

In [1]:
import os, sys
from pathlib import Path

# BLAS single-threaded before numpy/scipy load (matches the precompute scripts).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "BLIS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import numpy as np
import pandas as pd

# Put the repo root on sys.path so `scripts.*` and `kika.*` resolve.
_kika_root = Path.cwd()
while not (_kika_root / "kika").is_dir() and _kika_root != _kika_root.parent:
    _kika_root = _kika_root.parent
if str(_kika_root) not in sys.path:
    sys.path.insert(0, str(_kika_root))
print("repo root:", _kika_root)

from scripts.precompute_chi2_folded_c0 import load_library_folded, build_rows_at_energy
from scripts.precompute_chi2_library_c0 import interp_a_l_to_energy
from scripts.exfor_utils import build_exfor_cache_from_objects
from scripts.eval_covariance import build_eval_cov_for_groups
from scripts.tof_parameters import (
    load_tof_parameters_file, get_tof_parameters, compute_sigma_E,
)
from scripts import chi2_metrics
from kika.exfor import read_exfor

# --- Libraries (This_work = run 77; JEFF/JENDL as in the folded_c0 methodology) ---
LIB_FILES = {
    "JEFF":      "/share_snc/snc/JuanMonleon/jeff40_with_MF4_from_jeff33/26-Fe-56g.txt",
    "JENDL":     "/share_snc/snc/JuanMonleon/JENDL-5/260560.jendl5",
    "This_work": "/share_snc/snc/JuanMonleon/ENDF_samples/new_test_77/26-Fe-56g_nominal_mg.endf",  # run 77
}
LIB_DISPLAY = {"JEFF": "JEFF-4.0", "JENDL": "JENDL-5", "This_work": "This work"}
LIB_ORDER   = ["JEFF", "JENDL", "This_work"]

# --- Gkatis experiment (read directly from its JSON; not in the EXFOR DB) ---
GKATIS_ID   = "27673002"
GKATIS_JSON = "/share_snc/snc/JuanMonleon/EXFOR/data_v1/27673002.json"

# --- TOF energy resolution (delta_t treated as sigma via compute_sigma_E) ---
TOF_PARAMS_FILE = "/share_snc/snc/JuanMonleon/EXFOR/exfor_tof_parameters.json"

# --- Analysis window / truncation (must match the precompute module globals) ---
E_MIN_MEV, E_MAX_MEV = 0.85, 4.0
L_MAX, MT_NUMBER     = 6, 2

repo root: /home/MONLEON-JUAN/kika


## 1. Load the three libraries (MF3 elastic $\sigma$, MF4 $a_\ell$, MF34 covariance)

In [2]:
libraries = {
    "JEFF":      load_library_folded(LIB_FILES["JEFF"],      "JEFF-4.0"),
    "JENDL":     load_library_folded(LIB_FILES["JENDL"],     "JENDL-5"),
    "This_work": load_library_folded(LIB_FILES["This_work"], "This work (run 77)"),
}

Loading JEFF-4.0 from /share_snc/snc/JuanMonleon/jeff40_with_MF4_from_jeff33/26-Fe-56g.txt


  MF4: 3960 grid points, E=[1e-11, 45] MeV, max L=32
  MF3: 15048 pointwise σ values, E=[1e-11, 150] MeV


  MF34: 21 covariance blocks
Loading JENDL-5 from /share_snc/snc/JuanMonleon/JENDL-5/260560.jendl5


  MF4: 1796 grid points, E=[1e-11, 20] MeV, max L=24
  MF3: 19004 pointwise σ values, E=[1e-11, 200] MeV


  MF34: 1 covariance blocks
Loading This work (run 77) from /share_snc/snc/JuanMonleon/ENDF_samples/new_test_77/26-Fe-56g_nominal_mg.endf


  MF4: 4005 grid points, E=[1e-11, 45] MeV, max L=32
  MF3: 15048 pointwise σ values, E=[1e-11, 150] MeV


  MF34: 21 covariance blocks


## 2. Load Gkatis EXFOR data + TOF parameters, fold $c_0$ per energy

`build_rows_at_energy` folds each library's MF3 over the Gkatis resolution kernel
($c_0=\langle\sigma\rangle_\mathrm{fold}/4\pi$), interpolates $a_\ell$ (not folded), and
evaluates $y_\mathrm{eval}=\mathrm{legval}(\mu, [c_0, c_0(2\ell+1)a_\ell])$ per datapoint.

Gkatis is reported in the **LAB** frame, so `build_experiment_dataframe` transforms its
$\mu$, $\mathrm{d}\sigma/\mathrm{d}\Omega$, and uncertainties to the **CM** frame (the
frame of the MF4 Legendre coefficients) before the comparison.

In [3]:
tof_cache  = load_tof_parameters_file(TOF_PARAMS_FILE)
tof_gkatis = get_tof_parameters(GKATIS_ID, tof_cache)
print(f"Gkatis TOF: L={tof_gkatis.flight_path_m} m, dt={tof_gkatis.time_resolution_ns} ns "
      f"(source={tof_gkatis.source})")

# Gkatis lives only in its JSON, not the EXFOR database: read it directly.
# build_exfor_cache_from_objects applies the uncertainty manifest (sigma_stat /
# sigma_sys split), exactly as the full pipeline does.
gkatis_ad = read_exfor(GKATIS_JSON)
print(f"Loaded {gkatis_ad.label}: {len(gkatis_ad.energies())} incident energies")
exfor_cache, _sorted_e = build_exfor_cache_from_objects([gkatis_ad])

# Fold each library's MF3 over the Gkatis resolution at every incident energy.
energies_in_range = sorted(e for e in exfor_cache if E_MIN_MEV <= e <= E_MAX_MEV)
all_rows = []
for e_mev in energies_in_range:
    all_rows.extend(build_rows_at_energy(e_mev, exfor_cache[e_mev], libraries, tof_cache))

df = pd.DataFrame(all_rows)
df = df[df["experiment_id"] == GKATIS_ID].reset_index(drop=True)
print("rows:", len(df), "| per library:", df.groupby("library").size().to_dict())
print(f"energies: {df['energy_mev'].nunique()} unique, "
      f"[{df['energy_mev'].min():.4g}, {df['energy_mev'].max():.4g}] MeV")

Gkatis TOF: L=27.037 m, dt=10.0 ns (source=file)


Loaded Gkatis et al. (2025): 127 incident energies


  [uncertainty_manifest] 27673002 (manifest): σ_sys capped at σ_total on 1016/1016 rows; σ_stat floored at 1% of |y| on 1016/1016 rows.


rows: 2352 | per library: {'JEFF': 784, 'JENDL': 784, 'This_work': 784}
energies: 98 unique, [1.01, 3.934] MeV


### Folded $c_0$ diagnostics

Note: the pipeline modifies MF4/MF34 only, not MF3, so This work's folded $c_0$ equals
the base-evaluation $c_0$ &mdash; what distinguishes This work is its run-77 $a_\ell$ and MF34.

In [4]:
print(f"sigma_E(1 MeV) for Gkatis = {compute_sigma_E(1.0, tof_gkatis) * 1e3:.2f} keV\n")

rep = (df.groupby(["library", "energy_mev"], as_index=False)
         .agg(c0=("c0", "first"), sigma_avg_b=("sigma_avg_b", "first"),
              sigma_E_mev=("sigma_E_mev", "first")))
e_grid = sorted(rep["energy_mev"].unique())
sample_e = e_grid[:: max(1, len(e_grid) // 6)]
print("folded c0 (b/sr) per library:")
for e in sample_e:
    sub = rep[rep["energy_mev"] == e]
    c0s = ", ".join(f"{LIB_DISPLAY[r.library]}: {r.c0:.4f}" for r in sub.itertuples())
    sE = sub["sigma_E_mev"].iloc[0] * 1e3
    print(f"  E={e:6.4g} MeV (sigma_E={sE:5.1f} keV) | {c0s}")

sigma_E(1 MeV) for Gkatis = 10.23 keV

folded c0 (b/sr) per library:
  E=  1.01 MeV (sigma_E= 10.4 keV) | JEFF-4.0: 0.1784, JENDL-5: 0.1888, This work: 0.1784
  E= 1.197 MeV (sigma_E= 13.4 keV) | JEFF-4.0: 0.1830, JENDL-5: 0.1779, This work: 0.1830
  E=  1.44 MeV (sigma_E= 17.7 keV) | JEFF-4.0: 0.2124, JENDL-5: 0.2118, This work: 0.2124
  E= 1.767 MeV (sigma_E= 24.0 keV) | JEFF-4.0: 0.1650, JENDL-5: 0.1663, This work: 0.1650
  E= 2.219 MeV (sigma_E= 33.8 keV) | JEFF-4.0: 0.1792, JENDL-5: 0.1797, This work: 0.1792
  E=  2.87 MeV (sigma_E= 49.7 keV) | JEFF-4.0: 0.1859, JENDL-5: 0.1832, This work: 0.1859
  E= 3.856 MeV (sigma_E= 77.5 keV) | JEFF-4.0: 0.1827, JENDL-5: 0.1809, This work: 0.1827


## 3. Build $\Sigma_\mathrm{eval}$ (MF34) and compute the $\chi^2$ variants

`folded_c0` uses `systematic_block_col=None` (the folded normalization is a genuine
experiment-wide mode), matching `chi2_analysis_cluster.py`.

In [5]:
def _a_l_lookup(lib_key, lib, e_mev):
    return interp_a_l_to_energy(lib, e_mev, L_MAX)

eval_cov = build_eval_cov_for_groups(df, libraries, _a_l_lookup, l_max=L_MAX)

# sigma_eval diagonal (used by the diagnostic V1/V3 variants; V2/V4 do not need it).
sigma_eval_diag = np.zeros(len(df))
for (lib_key, exp_id), block in eval_cov.items():
    mask = (df["library"] == lib_key) & (df["experiment_id"] == exp_id)
    idx = np.flatnonzero(mask.to_numpy())
    sigma_eval_diag[idx] = np.sqrt(np.maximum(np.diag(block), 0.0))
df["sigma_eval_diag"] = sigma_eval_diag

chi2 = chi2_metrics.chi2_per_experiment_variants(
    df, eval_cov,
    group_cols=("library", "experiment_id"),
    systematic_block_col=None,
)
chi2[["library", "N", "chi2_v2_per_N", "chi2_v4_per_N"]]

,library,N,chi2_v2_per_N,chi2_v4_per_N
0,JEFF,784,728.864653,591.680698
1,JENDL,784,638.953287,623.433658
2,This_work,784,322.952281,245.069465


## 4. Comparison table &mdash; reduced $\chi^2/N$ on Gkatis 2025 (folded $c_0$)

In [6]:
by_lib = chi2.set_index("library")
table = pd.DataFrame({
    "Library": [LIB_DISPLAY[k] for k in LIB_ORDER],
    "N":       [int(by_lib.loc[k, "N"]) for k in LIB_ORDER],
    "sigma_exp (V2, EXFOR-only)":        [by_lib.loc[k, "chi2_v2_per_N"] for k in LIB_ORDER],
    "sigma_exp (+) sigma_eval (V4, EXFOR+MF34)": [by_lib.loc[k, "chi2_v4_per_N"] for k in LIB_ORDER],
})

# Markdown (copy-paste friendly).
cols = ["Library", "N", "$\\sigma_\\mathrm{exp}$ (V2)", "$\\sigma_\\mathrm{exp}\\oplus\\sigma_\\mathrm{eval}$ (V4)"]
lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
for k in LIB_ORDER:
    lines.append(f"| {LIB_DISPLAY[k]} | {int(by_lib.loc[k, 'N'])} | "
                 f"{by_lib.loc[k, 'chi2_v2_per_N']:.1f} | {by_lib.loc[k, 'chi2_v4_per_N']:.1f} |")
print("Reduced chi2/N on Gkatis 2025 (EXFOR 27673002), folded-c0 normalization:\n")
print("\n".join(lines))

table

Reduced chi2/N on Gkatis 2025 (EXFOR 27673002), folded-c0 normalization:

| Library | N | $\sigma_\mathrm{exp}$ (V2) | $\sigma_\mathrm{exp}\oplus\sigma_\mathrm{eval}$ (V4) |
|---|---|---|---|
| JEFF-4.0 | 784 | 728.9 | 591.7 |
| JENDL-5 | 784 | 639.0 | 623.4 |
| This work | 784 | 323.0 | 245.1 |


,Library,N,"sigma_exp (V2, EXFOR-only)","sigma_exp (+) sigma_eval (V4, EXFOR+MF34)"
0,JEFF-4.0,784,728.864653,591.680698
1,JENDL-5,784,638.953287,623.433658
2,This work,784,322.952281,245.069465


## 5. Sanity checks

In [7]:
N_by_lib = df.groupby("library").size()
assert N_by_lib.nunique() == 1, f"libraries have differing N: {N_by_lib.to_dict()}"
N = int(chi2["N"].iloc[0])
assert 700 <= N <= 820, f"unexpected N={N} (manuscript reports 784)"

sE = compute_sigma_E(1.0, tof_gkatis) * 1e3
assert 8.0 <= sE <= 12.0, f"sigma_E(1MeV)={sE:.2f} keV outside expected ~10 keV"

vals = chi2[["chi2_v2_per_N", "chi2_v4_per_N"]].to_numpy()
assert np.isfinite(vals).all() and (vals > 0).all(), "non-finite or non-positive chi2/N"

# Informational: is This work still the lowest under folded c0? (not asserted)
for v, name in [("chi2_v2_per_N", "V2"), ("chi2_v4_per_N", "V4")]:
    best = by_lib[v].idxmin()
    print(f"{name}: lowest chi2/N = {LIB_DISPLAY[best]} ({by_lib.loc[best, v]:.1f})")
print(f"\nN per library = {N}; sigma_E(1 MeV) = {sE:.2f} keV")
print("All structural sanity checks passed.")

V2: lowest chi2/N = This work (323.0)
V4: lowest chi2/N = This work (245.1)

N per library = 784; sigma_E(1 MeV) = 10.23 keV
All structural sanity checks passed.
